In [1]:
from pathlib import Path
import json
import sys
sys.path.append("..")  # Add above scripts directory to python modules path.
# import pipeline_mcorr_cnmf as pmc
import compute_zcorr as cz

path_file = Path('/home/wanglab/code/imaging_analysis/Analysis_2P/test/Paths_zmotion_lin2.json')

with open(path_file, 'r') as f:
    paths = json.load(f)

parameters_file = paths['params_files'][0]

with open(parameters_file, 'r') as f:
    parameters = json.load(f)
    
z_parameters_file = paths['z_params_files'][0]

with open(z_parameters_file, 'r') as f:
    z_parameters = json.load(f)

zstack_path = Path(paths['zstack_paths'][0])
z_shifted_file = z_parameters['zstack_shift']['file_name']

parameters['zstack_path'] = zstack_path
parameters['z_params'] = z_parameters

mesmerize_path = Path(paths['export_paths'][0])

mcorr_movie_path = mesmerize_path / 'd248ec2c-857b-4df3-a3ff-ba0b7043a153' / 'd248ec2c-857b-4df3-a3ff-ba0b7043a153-cat_tiff_bt_els__d1_765_d2_765_d3_1_order_F_frames_1600.mmap'

In [5]:
# zcorr_movie_path, _, _, patch_correlations_filepath = cz.z_motion(mcorr_movie_path, parameters)

In [ ]:
from caiman.mmapping import load_memmap
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib import cm
from scipy.io import savemat
from scipy.ndimage import gaussian_filter, map_coordinates, shift, affine_transform, label, find_objects
from scipy.stats import mode, linregress

In [ ]:
# _, patch_correlations_filepath = cz.correl_z_motion_patches(mcorr_movie_path, zstack_path / z_shifted_file, z_correlation, parameters['params_mcorr']['main'])
z_correlation=np.load(mesmerize_path / "z_correlation.npz")

In [ ]:
movie_mmap_path = mcorr_movie_path
zstack_filepath = zstack_path / z_shifted_file
# z_correlation
mcorr_params = parameters['params_mcorr']['main']
save_format= 'mat'

In [ ]:
# def correl_z_motion_patches(movie_mmap_path, zstack_filepath, z_correlation, mcorr_params, save_format='parquet'):
# """
# Find the correlation between patches in the F_func movie and the anat z-stack.
# """
# Time the execution
# import time
# start_time = time.time()

# Load the F functional movie
movie_16bit, dims, T = load_memmap(movie_mmap_path)
F_func = np.reshape(movie_16bit.T, [T] + list(dims), order='F')
# Clip the movie range to uint16
F_func = np.clip(F_func, 0, 2**16-1)
# Get dimensions    
Nframe, Ny, Nx = F_func.shape

# Load the FOV file
# mcorr_batch_dir = movie_mmap_path.parent
# fov_file_path = Path.joinpath(mcorr_batch_dir, f"{mcorr_batch_dir.stem}_mean_projection.npy")
# fov_image = np.load(fov_file_path)
# Instead create FOV
fov_image = np.mean(F_func, axis=0)

# Load the z-stack
info_zstack = Image.open(zstack_filepath)
Nz = info_zstack.n_frames
Zstack = np.zeros((Ny, Nx, Nz), dtype=np.float32)
for iz in range(Nz):
    info_zstack.seek(iz)
    Zstack[:, :, iz] = np.array(info_zstack)
info_zstack.close()        
# TODO: check why on Windows F_anat bit depth is 8 and not 16

# Get dimensions 
Zstack_shape = Zstack.shape
print(f"Zstack shape: {Zstack_shape}")

# Plot the z-stack's first frame
plt.imshow(Zstack[:, :, 0], cmap='gray')
plt.title('Z-stack first frame')
plt.show()


In [ ]:
# Flip Zstack up-down to match the orientation of the movie
Zstack = np.flip(Zstack, axis=0)

fig, ax = plt.subplots(1, 3, figsize=(10, 5))
# Plot the F_func frame
ax[0].imshow(F_func[0, :, :], cmap='gray')
ax[0].set_title('F_func frame')
# Plot the FOV
ax[1].imshow(fov_image, cmap='gray')
ax[1].set_title("FOV")
# Plot the Zstack frame
ax[2].imshow(Zstack[:, :, 0], cmap='gray')
ax[2].set_title('Zstack frame')
plt.show()

In [ ]:
# Define patch size based on Caiman's parameters. Width of patch = strides+overlaps.
# mcorr_params = parameters['params_mcorr']['main']
step_size = mcorr_params['strides'] 
patch_overlap = mcorr_params['overlaps']
# Calculate patch size for both x and y dimensions 
patch_size = [step + overlap for step, overlap in zip(step_size, patch_overlap)]

# Get zpos, x and y shifts
zpos = z_correlation['zpos']
xshift = z_correlation['xshift']
yshift = z_correlation['yshift']

In [ ]:
# Plot the zpos over time
plt.plot(zpos)
plt.xlabel('Frame number')
plt.ylabel('Z position')
plt.title('Z position over time')
plt.show()

In [ ]:
zcorrel = z_correlation['zcorr']

In [ ]:
# if patch_correlations.parquet exists, load it 
export_path = movie_mmap_path.parent.parent
# if (export_path / 'patch_correlations.parquet').exists():
#     patch_correlations_df = pd.read_parquet(export_path / 'patch_correlations.parquet')   

In [ ]:
def patch_regress(frame_data):
    """
    Calculate the linear least-squares regression between the F_func movie and the z-stack
    for each patch at several depths around zpos.

    Inputs: 
    - frame_data: tuple containing the frame number, the frame data, the z-stack, the x/y shifts, the dimensions of the movie, the patch size, the step size, the current zpos, and the valid indices.

    Outputs:
    - results: list of dictionaries containing the r_squared, slope, intercept, frame number, patch number, patch x/y limits, patch z index, and patch z position. 
    """

    np.seterr(over='raise')
        
    # fram_start_time = time.time()

    frameNum, frame_data, Zstack, frame_xshift, frame_yshift, Nx, Ny, patch_size, step_size, current_zpos, valid_indices = frame_data
    
    Z_frame_shifted = np.zeros_like(Zstack)
    patch_regress_results = []

    for z_idx in valid_indices:
        frameXshift = -frame_xshift[z_idx] # for no-shift control: 0 
        frameYshift = -frame_yshift[z_idx] # for no-shift control: 0
        transformation_matrix = np.array([[1, 0, frameYshift], [0, 1, frameXshift]])
        Z_frame_shifted[:, :, z_idx] = affine_transform(Zstack[:, :, z_idx], transformation_matrix, order=1, mode='constant', cval=0.0, prefilter=False)

    # Initialize patch number
    patch_num = -1
    
    # Loop over patches, moving vertically (along columns) first, then horizontally (along rows)    
    for i in range(0, Nx, step_size[1]):
        for j in range(0, Ny, step_size[0]):
            # Allow for a last partial patch on the right and bottom borders, but not more
            if i + patch_size[1] > Nx and patch_size_x < patch_size[1] and patch_size_y < patch_size[0]:
                continue 
            if j + patch_size[0] > Ny and patch_size_y < patch_size[0]:
                continue 
            # Adjust the patch size for patches on the bottom and right borders
            patch_size_x = patch_size[1] if i + patch_size[1] <= Nx else Nx - i
            patch_size_y = patch_size[0] if j + patch_size[0] <= Ny else Ny - j

            # Get the patch from the frame data
            T_patch = frame_data[j:j+patch_size_y, i:i+patch_size_x].ravel()
            # Increment patch number
            patch_num += 1
            
            # print(f"Frame {frameNum}, Patch {patch_num}, Patch Size: {patch_size_y}x{patch_size_x}")
            
            # Initialize results list
            patch_results = []
            # Loop over valid depth indices
            for z_idx in valid_indices:
                Z_patch = Z_frame_shifted[j:j+patch_size_y, i:i+patch_size_x, z_idx].ravel()
                if T_patch.size == Z_patch.size:
                    slope, intercept, r_value, _, _ = linregress(T_patch, Z_patch)
                    try:
                        patch_results.append({
                        'r_squared': r_value**2, 'slope': slope, 'intercept': intercept,
                        'frame_num': frameNum, 'patch_number': patch_num,
                        'patch_x_lims': [i, i+patch_size_x], 'patch_y_lims': [j, j+patch_size_y],
                        'patch_z_idx': int(z_idx) - int(current_zpos), 'patch_z_pos': z_idx
                        })
                    except FloatingPointError:
                        print("Overflow Error")
                        # print(f"z_idx: {z_idx}, current_zpos: {current_zpos}")
                        
            # Smooth R^2 values over valid depth indices
            r_squares = [result['r_squared'] for result in patch_results]
            r_squares = gaussian_filter(r_squares, sigma=1)
            # Get the max R^2 value's index
            max_r_square_idx = np.argmax(r_squares)
            # Append the result corresponding to the max R^2 value to patch_regress_results
            patch_regress_results.append(patch_results[max_r_square_idx])                      
                        
    # frame_end_time = time.time()
    # print(f"Frame {frameNum} processed in {frame_end_time - fram_start_time:.2f} seconds")
            
    return patch_regress_results

In [ ]:
# Done - Compare results with and without shifts
# Done - Run patch regress on mock Z-stack based on FOV (mean T-series) 
# Done - Run with +/- 5 z-indices 
# Done - Run with +/- 7 z-indices 
# Done - Compare results between linregress and corrcoef. Same results
# Run patch regress on single median z-stack frame (20)
# Done - Make a mock T-series with a z-stack frame. Should get ~perfect correlation. Confirmed

In [ ]:
# Create mock Z-stack with FOV. 
# Stack as many FOV images as there are z-indices in the z-stack.
fov_stack = np.stack([fov_image for _ in range(Zstack_shape[2])], axis=2)
print("FOV_stack", fov_stack.shape)

In [ ]:
# Create mock T-series with middle z-stack frame, and replicated with as many frames as the F_func movie.
mid_stack_frame = Zstack[:, :, Zstack_shape[2]//2]
T_series_mu = np.stack([mid_stack_frame for _ in range(Nframe)], axis=0)
print("T_series_mu", T_series_mu.shape)
print("T series shape", F_func.shape)

In [ ]:
# # Compute the patch correlations
# # Prepare data for parallel processing
# frame_data_list = []
# for frameNum in range(Nframe):
#     current_zpos = zpos[frameNum]
#     indices = [current_zpos - 3, current_zpos - 2, current_zpos - 1,
#             current_zpos, current_zpos + 1, current_zpos + 2, current_zpos + 3]
#     valid_indices = [index for index in indices if 0 <= index < Zstack_shape[2]]

#     frame_data_list.append((frameNum, F_func[frameNum, :, :], Zstack, xshift[:, frameNum], yshift[:, frameNum], Nx, Ny, patch_size, step_size, zpos[frameNum], valid_indices))
#     # Note: 
#     # Zstack may be replaced with fov_stack for testing
#     # F_func may be replaced with T_series_mu for testing
        
# # # Do regular for loop for debugging
# patch_regress_results = []
# for frame_data in tqdm(frame_data_list, desc="Find the correlation between patches in the F_func movie and the anat z-stack over frames"):
#     patch_regress_results.append(patch_regress(frame_data))
    
# # Flatten the list of lists
# patch_regress_results = [result for sublist in patch_regress_results for result in sublist]

# # Convert to DataFrame
# patch_correlations_df = pd.DataFrame(patch_regress_results)

In [ ]:
# if patch_correlations.parquet exists, load it 
export_path = movie_mmap_path.parent.parent
# if (export_path / 'patch_correlations_15zplanes.parquet').exists():
#     patch_correlations_df = pd.read_parquet(export_path / 'patch_correlations_15zplanes.parquet')
if (export_path / 'patch_correlations.parquet').exists():
    patch_correlations_df = pd.read_parquet(export_path / 'patch_correlations.parquet')        
else: 
    # if not, compute the patch correlations
    start_time = time.time()
    # Prepare data for parallel processing
    frame_data_list = []
    for frameNum in range(Nframe):
        current_zpos = zpos[frameNum]
        
        # indices = [current_zpos]
        # The number of z-stack frames to consider is 11 (+/- 5 frames around the current zpos)
        indices = [current_zpos + i for i in range(-5, 6)]
        
        valid_indices = [index for index in indices if 0 <= index < Zstack_shape[2]]
        
        frame_data_list.append((frameNum, F_func[frameNum, :, :], Zstack, xshift[:, frameNum], yshift[:, frameNum], Nx, Ny, patch_size, step_size, zpos[frameNum], valid_indices))
        # Note: 
        # Zstack may be replaced with fov_stack for testing
        # F_func may be replaced with T_series_mu for testing
    
    # Use Parallel to parallelize the task 
    patch_regress_results = Parallel(n_jobs=-1)(delayed(patch_regress)(frame_data) for frame_data in tqdm(frame_data_list,
                                desc="Find the correlation between patches in the F_func movie and the anat z-stack over frames"))

    # Flatten list of lists returned by map
    patch_correlations = [item for sublist in patch_regress_results for item in sublist]

    # Convert to DataFrame
    patch_correlations_df = pd.DataFrame(patch_correlations)

    # Display execution time in mn and seconds
    exect_time = time.time() - start_time
    print("Correlation computation took: ", exect_time // 60, "mn and ", exect_time % 60, "seconds")
    
# For 1600 frames, correlation computation took:  2.0 mn and  36.99588894844055 seconds

In [ ]:
patch_correlations_filepath = export_path / 'patch_correlations.parquet'
# patch_correlations_filepath = export_path / 'patch_correlations_single_zpos_control.parquet'
patch_correlations_df.to_parquet(patch_correlations_filepath, index=False)
# patch_correlations_control_df = patch_correlations_df

patch_correlations_filepath = export_path / 'patch_correlations.csv'
patch_correlations_df.to_csv(patch_correlations_filepath, index=False)

In [ ]:
# Get row [49463] from patch_correlations_df
row_vals = patch_correlations_df.iloc[49463]
print(row_vals)

In [ ]:
print(f"F_func dimensions are ordered as Frames (T), Y, X: {F_func.shape}")
print(f"Zstack dimensions are ordered as Y, X, Z: {Zstack.shape}")

In [ ]:
# Plot the F_func frame, and the Z-stack frame, with the patch outline overlayed in red
F_func_frame = F_func[row_vals['frame_num'], :, :]
Z_stack_frame = Zstack[:, :, row_vals['patch_z_pos']]
# Create a colormap for the Z-stack frame
cmap = plt.cm.gray
cmaplist = [cmap(i) for i in range(cmap.N)]
cmaplist[0] = (0.0, 0.0, 0.0, 1.0)
cmap = cmap.from_list('Custom cmap', cmaplist, cmap.N)
# Create a figure
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
# Plot the F_func frame
ax[0].imshow(F_func_frame, cmap='gray')
# Plot the Z-stack frame
ax[1].imshow(Z_stack_frame, cmap=cmap)
# Draw the patch outline in red
patch_x_lims = row_vals['patch_x_lims']
patch_y_lims = row_vals['patch_y_lims']
patch_x = [patch_x_lims[0], patch_x_lims[1], patch_x_lims[1], patch_x_lims[0], patch_x_lims[0]]
patch_y = [patch_y_lims[0], patch_y_lims[0], patch_y_lims[1], patch_y_lims[1], patch_y_lims[0]]
ax[0].plot(patch_y, patch_x, color='red')
ax[1].plot(patch_y, patch_x, color='red')
# plt.show() 

In [ ]:
row_nums = [49462, 49463, 49464]
fig, ax = plt.subplots(3, 2, figsize=(10, 7)) 

plt.subplots_adjust(wspace=1, hspace=0.5) 

# Plot three consecutive patches for F_func (left column) and F_anat (right column)
for i, row_num in enumerate(row_nums):
    row_vals = patch_correlations_df.iloc[row_num]

    F_func_frame = F_func[row_vals['frame_num'], :, :]
    Z_stack_frame = Zstack[:, :, row_vals['patch_z_pos']]

    F_func_patch_data = F_func_frame[row_vals['patch_y_lims'][0]:row_vals['patch_y_lims'][1], row_vals['patch_x_lims'][0]:row_vals['patch_x_lims'][1]]
    F_anat_patch_data = Z_stack_frame[row_vals['patch_y_lims'][0]:row_vals['patch_y_lims'][1], row_vals['patch_x_lims'][0]:row_vals['patch_x_lims'][1]]
    
    # Plot the patch from the F_func movie and Z-stack side by side
    ax[i, 0].imshow(F_func_patch_data, cmap='gray')  
    # remove the x and y ticks and labels
    ax[i, 0].set_xticks([])
    ax[i, 0].set_yticks([])
    ax[i, 0].set_title(f"patch_x_lims: {row_vals['patch_x_lims']}, patch_y_lims: {row_vals['patch_y_lims']}, \n \
                        frame number: {row_vals['frame_num']}, $R^2$: {row_vals['r_squared']:.2f}")
    ax[i, 1].imshow(F_anat_patch_data, cmap='gray')  
    ax[i, 1].set_xticks([])
    ax[i, 1].set_yticks([])
    ax[i, 1].set_title(f"Z-stack patch at z index: \n \
        {row_vals['patch_z_idx']}, z position: {row_vals['patch_z_pos']}")
    
plt.show()

In [ ]:
row_nums = [49462, 49463, 49464]

for row_num in row_nums:
    row_vals = patch_correlations_df.iloc[row_num]
    patch_x_lims = row_vals['patch_x_lims']
    patch_y_lims = row_vals['patch_y_lims']
    print(f"Patch {row_vals['patch_number']} at frame {row_vals['frame_num']} has x limits: {patch_x_lims}, y limits: {patch_y_lims}, \n \
            z index: {row_vals['patch_z_idx']}, z position: {row_vals['patch_z_pos']}, and R^2: {row_vals['r_squared']:.2f}")

In [ ]:
row_nums = [49462, 49463, 49464]

# Create a colormap for the Z-stack frame
cmap = plt.cm.gray
cmaplist = [cmap(i) for i in range(cmap.N)]
cmaplist[0] = (0.0, 0.0, 0.0, 1.0)
cmap = cmap.from_list('Custom cmap', cmaplist, cmap.N)

# Plot the F_func frame, and the Z-stack frame, with the patch outline overlayed in red
for row_num in row_nums:
    row_vals = patch_correlations_df.iloc[row_num]
    F_func_frame = F_func[row_vals['frame_num'], :, :]
    Z_stack_frame = Zstack[:, :, row_vals['patch_z_pos']]

    # Create a figure
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    # Plot the F_func frame
    ax[0].imshow(F_func_frame, cmap='gray')
    # Plot the Z-stack frame
    ax[1].imshow(Z_stack_frame, cmap=cmap)
    # Draw the patch outline in red
    patch_x_lims = row_vals['patch_x_lims']
    patch_y_lims = row_vals['patch_y_lims']
    patch_x = [patch_x_lims[0], patch_x_lims[1], patch_x_lims[1], patch_x_lims[0], patch_x_lims[0]]
    patch_y = [patch_y_lims[0], patch_y_lims[0], patch_y_lims[1], patch_y_lims[1], patch_y_lims[0]]
    ax[0].plot(patch_x, patch_y, color='red')
    ax[1].plot(patch_x, patch_y, color='red')
    plt.show()

In [ ]:
def calculate_zones(patch_correlations_df, Ny, Nx):
    """
    Find zones (i.e., how patches overlap) in the patch_correlations_df DataFrame.
    """
    # Initialize a mask for zone calculation
    mask = np.zeros((Ny, Nx), dtype=int)

    # Use the first frame's patches to define zones
    first_frame_patches = patch_correlations_df[patch_correlations_df['frame_num'] == patch_correlations_df['frame_num'].min()]
    for index, row in first_frame_patches.iterrows():
        x_start, x_end = row['patch_x_lims']
        y_start, y_end = row['patch_y_lims']
        mask[y_start:y_end, x_start:x_end] += 1
    
    # Label zones based on overlapping patches. Assign a unique number to each zone

    # Initialize the zone_pattern array to store zone identifiers
    zone_pattern = np.zeros((Ny, Nx), dtype=int)

    # Change the mask values from 4 to 3 (even to odd)
    mask = np.where(mask == 4, 3, mask)
    # Process for odd values (mask values that are odd become 1, others 0)
    binary_mask_odd = (mask % 2 == 1).astype(int)
    labeled_zones_odd, num_zones_odd = label(binary_mask_odd)

    # Assign unique labels to zone_pattern for odd zones
    zone_id_offset = 0  # Start zone IDs from 1 for odd zones
    for i in range(1, num_zones_odd + 1):
        zone_pattern[labeled_zones_odd == i] = zone_id_offset + i

    # Update the offset for even zones to continue unique labeling
    zone_id_offset += num_zones_odd

    # Process for even values (mask values that are even and non-zero become 1, others 0)
    binary_mask_even = ((mask % 2 == 0) & (mask != 0)).astype(int)
    labeled_zones_even, num_zones_even = label(binary_mask_even)

    # Assign unique labels to zone_pattern for even zones
    for i in range(1, num_zones_even + 1):
        zone_pattern[labeled_zones_even == i] = zone_id_offset + i

    # Re-label zones to have continuous IDs
    # unique_zones = np.unique(zone_pattern)  # Get all unique zone IDs

    # Flatten the array column-wise
    zone_pattern_col_flat = zone_pattern.flatten(order='F')

    # Get the unique zones and their unique indices
    unique_zones, inverse_indices = np.unique(zone_pattern_col_flat, return_inverse=True)
    unique_indices = pd.unique(inverse_indices) + 1
    
    # # Reshape the inverse indices back to the original shape of zone_pattern
    # zone_pattern_contig = inverse_indices.reshape(zone_pattern.shape, order='F')
    
    zone_pattern_contig = np.zeros_like(zone_pattern)
    for new_id, zone_id  in enumerate(unique_indices):
        zone_pattern_contig[zone_pattern == zone_id] = new_id

    # Create a look-up table. For each pixel coordinates, get the zone ID from zone_pattern_contig
    lookup_table = {index: value for index, value in np.ndenumerate(zone_pattern_contig)}
    
    # # Plot the zone pattern, no tick or label, with one segmentation color for each different pixel value.
    # plt.imshow(zone_pattern, cmap='plasma')
    # plt.xticks([])
    # plt.yticks([])
    # # plt.show()
    # # Save to export path / plots
    # plot_export_path = export_path / 'plots'
    # plot_export_path.mkdir(parents=True, exist_ok=True)
    # plt.savefig(plot_export_path / 'zone_pattern.png')
    
    return zone_pattern_contig, lookup_table, zone_pattern

In [ ]:
labeled_zones, lookup_table, zone_pattern= calculate_zones(patch_correlations_df, Ny, Nx)

In [ ]:
def average_patch_correlations(patch_correlations_df, labeled_zones):
    fields = ['r_squared', 'slope', 'intercept', 'patch_z_idx', 'patch_z_pos']

    # labeled_zones = calculate_zones_once(patch_correlations_df, Nx, Ny)

    zone_data = []

    # Iterate over frames
    for frame_num, group in patch_correlations_df.groupby('frame_num'):
        sums = {field: np.zeros((np.max(labeled_zones)+1,)) for field in fields}
        count = np.zeros((np.max(labeled_zones)+1,))

        # Populate sums and counts per zone
        for index, row in group.iterrows():
            x_start, x_end = row['patch_x_lims']
            y_start, y_end = row['patch_y_lims']
            # print(f"Frame {frame_num}, Patch {row['patch_number']}, x_start: {x_start}, x_end: {x_end}, y_start: {y_start}, y_end: {y_end}")
            
            zone_ids = np.unique(labeled_zones[y_start:y_end, x_start:x_end])
            # print(f"Zone IDs: {zone_ids}")
            
            # Get 'r_squared' values for group rows with group['patch_number'] == 0, 1, 21, 22
            # corner_r_squared = group.loc[(group['patch_number'] == 0) | (group['patch_number'] == 1) | (group['patch_number'] == 21) | (group['patch_number'] == 22), 'r_squared']
            # print(f"Corner R^2 values: \n{corner_r_squared} \nAverage: {corner_r_squared.mean()}")
            
            for zone_id in zone_ids:
                # print(f"Zone ID: {zone_id}")
                for field in fields:
                    sums[field][zone_id] += row[field]
                    # print(f"Field: {field}, Sum: {sums[field][zone_id]}")
                count[zone_id] += 1

        # Calculate averages for each zone
        averages = {field: np.zeros((np.max(labeled_zones)+1,)) for field in fields}
        for field in sums:
            averages[field] = np.divide(sums[field], count, where=(count > 0))

        # Collect data per zone
        for zone_id in range(0, np.max(labeled_zones)+1):
            zone_info = {'zone_id': zone_id, 'frame_num': frame_num}
            if count[zone_id] > 0:
                for field in fields:
                    zone_info[field] = averages[field][zone_id]
                zone_data.append(zone_info)
                
        # # plot zone_data's average r_squared values as a heatmap
        # r_squared_values = [zone['r_squared'] for zone in zone_data]
        # r_squared_values = np.array(r_squared_values)
        # r_squared_values = r_squared_values.reshape((41, 41), order='F')
        # plt.imshow(r_squared_values, cmap='viridis')
        # plt.colorbar()
        # plt.title(f"Average $R^2$ values per zone for frame {frame_num}")
        # # plt.show()
        # plot_export_path = export_path / 'plots'
        # plot_export_path.mkdir(parents=True, exist_ok=True)
        # plt.savefig(plot_export_path / 'r_square_heatmap.png')

    zone_df = pd.DataFrame(zone_data)
    return zone_df

In [ ]:
# Iterate over overlapping regions. For pixels with overlap between patches, average the correlation values.
# labeled_zones, zone_df = cz.average_patch_correlations(patch_correlations_df, Nx, Ny)
zone_df = average_patch_correlations(patch_correlations_df, labeled_zones)

### Make summary plots and stats

In [ ]:
# Plot distribution of correlation for all patches across depth, to see if we need to correlate with more z-stack planes
# This should be done on patch-regress results where all z correlation values are stored. 
# Here since we only have the best z plane, we can plot the distribution of patch_z_idx values

# patch_z_idx = patch_correlations_df['patch_z_idx']
# patch_z_idx = patch_correlations_11zplanes_df['patch_z_idx'] 
# patch_z_idx = patch_correlations_15zplanes_df['patch_z_idx'] 
# patch_z_idx = patch_correlations_fov_df['patch_z_idx'] 
patch_z_idx = patch_correlations_zstack_df['patch_z_idx'] 

# The number of bins will be the number of indices (e.g. 7, for -3 to 3 around zpos)
#  Get that number from the unique values of patch_z_idx
bin_num = len(np.unique(patch_z_idx))
# Define bin edges based on bin_num
bin_edges = np.arange(-bin_num/2, bin_num//2 + 1, 1)
plt.hist(patch_z_idx, bins=bin_edges, edgecolor='black')
plt.xlabel('Patch z index')
plt.ylabel('Frequency')
plt.title('Distribution of patch z indices, w/r to zpos')
# Set x-ticks to be the integer values
plt.xticks(np.arange(-bin_num//2 + 1, bin_num//2 + 1, 1))
plt.show()

In [ ]:
# patch_correlations_df = pd.read_parquet(export_path / 'patch_correlations.parquet')

In [ ]:
# Look at difference in R2 for each patch with or without correction (e.g., set best z plane to zpos for control) 
patch_rsquare_vals = patch_correlations_df['r_squared']
# control_patch_rsquare_vals = patch_correlations_control_df['r_squared']
# patch_rsquare_vals = patch_correlations_control_df['r_squared']
# control_patch_rsquare_vals = patch_correlations_revshift_df['r_squared']
# control_patch_rsquare_vals = patch_correlations_noshift_df['r_squared']
# patch_rsquare_vals = patch_correlations_15zplanes_df['r_squared']
# control_patch_rsquare_vals = patch_correlations_df['r_squared']
# control_patch_rsquare_vals = patch_correlations_fov_df['r_squared']
control_patch_rsquare_vals = patch_correlations_zstack_df['r_squared']

#  Remove values below 0.1
# patch_rsquare_vals = patch_rsquare_vals[patch_rsquare_vals > 0.1]
# control_patch_rsquare_vals = control_patch_rsquare_vals[control_patch_rsquare_vals > 0.1]

# Calculate mean R^2 values for corrected and uncorrected data
mean_uncorrected = control_patch_rsquare_vals.mean()
mean_corrected = patch_rsquare_vals.mean()
# Calculate the difference
diff = mean_corrected - mean_uncorrected
# Express the difference as a percentage of the uncorrected mean
improvement_percentage = (diff / mean_uncorrected) * 100

print(f"The z-correction improved the mean R^2 value by {improvement_percentage:.2f}%")

# Create subplots
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

# Plot the distribution of R^2 values for each, on the left subplot
c_label = 'T-series made of mid Z-stack frame' #'single zpos (zpos "0")' 'single zpos, no shift' 'single zpos, inverse shift' 'single zpos (zpos "0")''+/- 3 planes around zpos 0'
t_label =  '+/- 5 planes around zpos 0'#'single zpos, shift''+/- 3 planes around zpos 0'
ax[0].hist(control_patch_rsquare_vals, bins=50, alpha=0.5, label=c_label)
ax[0].hist(patch_rsquare_vals, bins=50, alpha=0.5, label=t_label)
# Add legend
ax[0].legend(loc='upper right')
ax[0].set_xlabel('$R^2$ value')
ax[0].set_ylabel('Frequency')
ax[0].set_title('Distribution of $R^2$ values for patches')

# Plot the cumulative distribution of R^2 values for each, on the right subplot
ax[1].hist(control_patch_rsquare_vals, bins=50, alpha=0.5, label=c_label, cumulative=True, histtype='step')
ax[1].hist(patch_rsquare_vals, bins=50, alpha=0.5, label=t_label, cumulative=True, histtype='step')
# Add legend
ax[1].legend(loc='upper left')
ax[1].set_xlabel('$R^2$ value')
ax[1].set_ylabel('Frequency')
ax[1].set_title(f"Cumulative distribution of $R^2$ values. Improvement: {improvement_percentage:.2f}%")

# Display the plot
plt.tight_layout()
plt.show()


In [ ]:
zcorrel = z_correlation['zcorr']
# Get max value for each column
max_vals = np.max(zcorrel, axis=0)
rsquare_corr = np.square(max_vals)

# Plot the distribution
plt.hist(rsquare_corr, bins=50, edgecolor='black')
plt.xlabel('$R^2$ value')
plt.ylabel('Frequency')
plt.title('Distribution of $R^2$ values from compute_zcorrel (whole frames)')

plt.show()

In [ ]:
# Calculate cumulative histograms
hist_uncorrected, bin_edges = np.histogram(control_patch_rsquare_vals, bins=50, density=True)
hist_corrected, _ = np.histogram(patch_rsquare_vals, bins=bin_edges, density=True)

# Calculate cumulative sums
cumulative_uncorrected = np.cumsum(hist_uncorrected)/np.sum(hist_uncorrected)
cumulative_corrected = np.cumsum(hist_corrected)/np.sum(hist_corrected)

# Calculate the difference between the two cumulative curves
diff_cumulative = cumulative_corrected - cumulative_uncorrected

# You can then calculate the mean of the absolute differences
mean_diff = np.mean(np.abs(diff_cumulative))

print(f"The difference between the cumulative distributions is {mean_diff:.2f}")


In [ ]:
# Patch overlap pattern
# Plot the zone pattern, no tick or label, with one segmentation color for each different pixel value.
plt.imshow(zone_pattern, cmap='plasma')
plt.xticks([])
plt.yticks([])
plt.show()
# Save to export path / plots
# plot_export_path = export_path / 'plots'
# plot_export_path.mkdir(parents=True, exist_ok=True)
# plt.savefig(plot_export_path / 'zone_pattern.png')

In [ ]:
# R^2 heatmap
# plot zone_data's average r_squared values as a heatmap
dim_num_zones = int(np.sqrt(np.max(labeled_zones) + 1))

# Get zone_df data corresponding to the first frame
frame_num = zone_df['frame_num'].min()
zone_data = zone_df[zone_df['frame_num'] == frame_num]

r_squared_values = zone_data['r_squared']
r_squared_values = np.array(r_squared_values)

r_squared_values = r_squared_values.reshape((dim_num_zones, dim_num_zones), order='F')
plt.imshow(r_squared_values, cmap='viridis')
plt.colorbar()
plt.title(f"Average $R^2$ values per zone for frame {frame_num} \n \
    (each zone plotted as a single pixel)")
plt.show()
# plot_export_path = export_path / 'plots'
# plot_export_path.mkdir(parents=True, exist_ok=True)
# plt.savefig(plot_export_path / 'r_square_heatmap.png')

In [ ]:
#  Loop over each zones' pixel region and get the average fluorescence value for each zone from the FOV image
zone_mean_fluorescence = []
zone_rsquare = []
for zone_id in range(0, np.max(labeled_zones)+1):
    zone_info = {'zone_id': zone_id, 'frame_num': frame_num}
    zone_indices = np.where(labeled_zones == zone_id)
    zone_fluorescence = fov_image[zone_indices]
    zone_mean_fluorescence.append(np.mean(zone_fluorescence)) 
    # Get the R2 value for that zone
    zone_info['r_squared'] = zone_data[zone_data['zone_id'] == zone_id]['r_squared'].values[0]
    zone_rsquare.append(zone_info['r_squared']) 
    

In [ ]:
# Plot the curve of average R2 values as a function of fluorescence using zone_rsquare and zone_mean_fluorescence
plt.scatter(zone_mean_fluorescence, zone_rsquare, alpha=0.5, edgecolor='black', s=10, facecolors='none', marker='o')
plt.xlabel('Average Fluorescence')
plt.ylabel('Average $R^2$')
plt.title('Average $R^2$ vs Average Fluorescence')
plt.show()

In [ ]:
import matplotlib.patches as patches

zone_labels = np.unique(labeled_zones)
# Get a zone with an average fluorescence
overall_average_fluorescence = np.mean(zone_mean_fluorescence)
# Find the zone whose mean fluorescence is closest to the overall average fluorescence
average_fluorescence_zone = zone_labels[np.argmin(np.abs(np.array(zone_mean_fluorescence) - overall_average_fluorescence))]

max_fluorescence_zone = np.argmax(zone_mean_fluorescence)

zone_id = max_fluorescence_zone # average_fluorescence_zone #716
zone_indices = np.where(labeled_zones == zone_id)

# Calculate bounding box corners
min_row, max_row = np.min(zone_indices[0]), np.max(zone_indices[0])
min_col, max_col = np.min(zone_indices[1]), np.max(zone_indices[1])

# Plot the FOV image
fig, ax = plt.subplots()
ax.imshow(fov_image, cmap='gray')

# Create a red rectangle patch
rect = patches.Rectangle((min_col, min_row), max_col-min_col+1, max_row-min_row+1, linewidth=1, edgecolor='r', facecolor='none')

# Add the patch to the Axes
ax.add_patch(rect)
plt.axis('off')  # Turn off axis
plt.title(f"FOV with Zone {zone_id} Highlighted")
plt.show()

# Get the fluorescence values for the zone
zone_fluorescence = fov_image[zone_indices]

# Create an empty 2D array for the fluorescence image
min_row, max_row = zone_indices[0].min(), zone_indices[0].max()
min_col, max_col = zone_indices[1].min(), zone_indices[1].max()

# Initialize an array to hold the reshaped zone fluorescence
fluorescence_image = np.zeros((max_row - min_row + 1, max_col - min_col + 1))

# Populate the fluorescence image
for i, row in enumerate(zone_indices[0]):
    col = zone_indices[1][i]
    fluorescence_image[row - min_row, col - min_col] = zone_fluorescence[i]

# Plot the zone fluorescence
plt.imshow(fluorescence_image, cmap='gray')
plt.axis('off')
plt.title(f"Zone {zone_id} Fluorescence")
plt.show()


In [ ]:
if save_format == 'mat':
# # Save the patch_correlations_df to a table in a .mat file
# # Convert DataFrame to a dictionary
# data_dict = {col: patch_correlations_df[col].values for col in patch_correlations_df.columns}
# savemat(mesmerize_path / 'patch_correlations_dict.mat', {'patch_correlations': data_dict})
    patch_correlations_filepath = export_path / 'patch_correlations.mat'
    zone_df_mat = cz.format_patch_correl_for_mat(zone_df)
    savemat(patch_correlations_filepath, zone_df_mat)

elif save_format == 'parquet':
    # Save the patch_correlations_df to a Parquet file
    patch_correlations_filepath = export_path / 'patch_correlations.parquet'
    patch_correlations_df.to_parquet(patch_correlations_filepath, index=False)

elif save_format == 'csv':
    # Save the patch_correlations_df to a csv file
    patch_correlations_filepath = export_path / 'patch_correlations.csv'
    patch_correlations_df.to_csv(patch_correlations_filepath, index=False)

# return patch_correlations_df, patch_correlations_filepath